# AG-UI 1.0 event translation — fully offline (Phase PRT)

> Notebook generated from [`examples/agui_translate.py`](https://github.com/prismal-ai/prismal/blob/main/examples/agui_translate.py).

| Field | Value |
|------|-------|
| **API key** | 🔑 Not required — the `astream` chunks are hand-built |
| **Source** | `examples/agui_translate.py` |
| **Guide** | [`docs/protocols-2026.md`](https://github.com/prismal-ai/prismal/blob/main/docs/protocols-2026.md) |

`prismal.events.agui` is a **pure translator**: it turns each item of
`graph.astream(input, config, stream_mode=["messages", "updates", "custom"])` into AG-UI 1.0 events. It has no
HTTP code (an AST guard enforces that) and never raises — the host (`prismal-server`) owns the transport.

AG-UI 1.0 has no `USAGE` or `INTERRUPT` events, so usage travels in `RUN_FINISHED.usage[]` plus a
`CUSTOM prismal.usage` event, and interrupts in `RUN_FINISHED.outcome`. Since 3.20.0, Skynet workers and MoA
proposers emit `subagent` custom payloads that become `SUBAGENT_STARTED / FINISHED / ERROR`.

In a real host:

```python
ctx = AgUiContext(run_id=run_id, thread_id=thread_id)
frames = [to_sse(e) for e in start_run(ctx, initial_state=state)]
async for chunk in graph.astream(state, config, stream_mode=["messages", "updates", "custom"]):
    frames += [to_sse(e) for e in to_agui_events(chunk, ctx)]
frames += [to_sse(e) for e in finish_run(ctx, summary=get_run_summary(thread_id))]
```

In [1]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and a fake `astream`

In [2]:
from __future__ import annotations

from collections import Counter
from typing import Any

from langchain_core.messages import AIMessageChunk, ToolMessage

from prismal.budget.types import Usage
from prismal.events.agui import AgUiContext, finish_run, start_run, to_agui_events, to_sse
from prismal.monitoring.observability_types import RunSummary


def fake_astream() -> list[Any]:
    """What ``graph.astream(stream_mode=["messages", "updates", "custom"])`` would yield."""
    meta = {"langgraph_node": "agent"}
    return [
        ("updates", {"supervisor": {"next_agent": "agent", "metadata": {"routing": {"tier": "fast"}}}}),
        (
            "messages",
            (
                AIMessageChunk(
                    content="",
                    id="msg-1",
                    tool_call_chunks=[
                        {"id": "call-1", "name": "web_search", "args": '{"q": "ag-ui"}', "index": 0}
                    ],
                ),
                meta,
            ),
        ),
        ("messages", (ToolMessage(content="3 results", tool_call_id="call-1", id="tool-1"), meta)),
        ("custom", {"type": "subagent", "phase": "start", "id": "w1", "name": "skynet_worker"}),
        ("custom", {"type": "subagent", "phase": "end", "id": "w1", "result": "ok"}),
        ("messages", (AIMessageChunk(content="AG-UI is ", id="msg-2"), meta)),
        ("messages", (AIMessageChunk(content="an event protocol.", id="msg-2"), meta)),
        ("updates", {"agent": {"current_agent": "agent"}}),
    ]

## 1 · Translate chunk by chunk

The `AgUiContext` keeps the small amount of state the translator needs (open messages, open tool calls, steps).

In [3]:
ctx = AgUiContext(run_id="run-1", thread_id="thread-1")
events = start_run(ctx, initial_state={"current_agent": "supervisor"})
for chunk in fake_astream():
    produced = to_agui_events(chunk, ctx)
    print(f"{chunk[0]:8} -> {[e.type.value for e in produced]}")
    events.extend(produced)

updates  -> ['STEP_STARTED', 'STATE_DELTA', 'CUSTOM', 'STEP_FINISHED']
messages -> ['TOOL_CALL_START', 'TOOL_CALL_ARGS']
messages -> ['TOOL_CALL_END', 'TOOL_CALL_RESULT']
custom   -> ['SUBAGENT_STARTED']
custom   -> ['SUBAGENT_FINISHED']
messages -> ['TEXT_MESSAGE_START', 'TEXT_MESSAGE_CONTENT']
messages -> ['TEXT_MESSAGE_CONTENT']
updates  -> ['TEXT_MESSAGE_END', 'STEP_STARTED', 'STATE_DELTA', 'STEP_FINISHED']


## 2 · Finish the run with a usage summary

`RunSummary.to_usage_wire()` is the shared 12-key usage wire (contract fixture shared with prismal-server and the SDK).

In [4]:
summary = RunSummary(
    run_id="run-1",
    session_id="thread-1",
    agent_name="supervisor",
    visited_nodes=["supervisor", "agent"],
    spans=[],
    tool_calls=[],
    usage=Usage(prompt_tokens=120, completion_tokens=30, calls=2, cost_source="table"),
    latency_ms=850.0,
)
events.extend(finish_run(ctx, summary=summary))
print("usage wire:", summary.to_usage_wire())

usage wire: {'prompt_tokens': 120, 'completion_tokens': 30, 'cache_read_tokens': 0, 'cache_write_tokens': 0, 'reasoning_tokens': 0, 'total_tokens': 150, 'calls': 2, 'cost_usd': 0.0, 'cost_source': 'table', 'estimated': True, 'served_model': None, 'gateway': None}


## 3 · Serialize to SSE frames

In [5]:
frames = [to_sse(e) for e in events]
print(Counter(e.type.value for e in events))
print()
print("".join(frames))

Counter({'STEP_STARTED': 2, 'STATE_DELTA': 2, 'CUSTOM': 2, 'STEP_FINISHED': 2, 'TEXT_MESSAGE_CONTENT': 2, 'RUN_STARTED': 1, 'STATE_SNAPSHOT': 1, 'TOOL_CALL_START': 1, 'TOOL_CALL_ARGS': 1, 'TOOL_CALL_END': 1, 'TOOL_CALL_RESULT': 1, 'SUBAGENT_STARTED': 1, 'SUBAGENT_FINISHED': 1, 'TEXT_MESSAGE_START': 1, 'TEXT_MESSAGE_END': 1, 'RUN_FINISHED': 1})

data: {"type":"RUN_STARTED","timestamp":1793671527569,"threadId":"thread-1","runId":"run-1","protocolVersion":"1.0"}

data: {"type":"STATE_SNAPSHOT","timestamp":1793671527570,"snapshot":{"current_agent":"supervisor"}}

data: {"type":"STEP_STARTED","timestamp":1793671527584,"stepName":"supervisor"}

data: {"type":"STATE_DELTA","timestamp":1793671527584,"delta":[{"op":"add","path":"/next_agent","value":"agent"}]}

data: {"type":"CUSTOM","timestamp":1793671527584,"name":"prismal.routing","value":{"tier":"fast"}}

data: {"type":"STEP_FINISHED","timestamp":1793671527584,"stepName":"supervisor"}

data: {"type":"TOOL_CALL_START","timestamp":17936715275

---

The same phase also ships the A2A v0.3 + v1.0 dual-stack, the MCP 2026-07-28 stateless wire with the tasks extension, and the read-only host surface on `RuntimeContext` (`budget_status`, `quota_windows`, `audit_page`, `decisions_report`, `routing_report`, …). Wire names are pinned in `docs/adr/ADR-PRT-001-protocol-pins.md`.